# Clase 4 · Partir el mapa en regiones

**Machine Learning** · Programa de Especialización Data Scientist · Módulo 1 · Sesión 4 de 16

Docente: **Josef Renato Rodriguez**

---

## De la Clase 3 a la Clase 4

Seguimos con **California Housing** y el mismo objetivo: predecir el valor mediano de la vivienda.
El SVR con kernel RBF nos mostró que había mucha precisión escondida en las **curvas**: subió el R² de
0.651 a 0.759. Hoy buscamos esa precisión de otra forma: con **árboles**, que no trazan
rectas ni curvas, sino que **parten los datos en regiones** y predicen el promedio de cada una.

| Modelo de hoy | La idea | Lo que resuelve |
|---|---|---|
| **Árbol de regresión** | preguntas de sí o no, una tras otra | curvas e interacciones, sin escalar nada |
| **Random Forest** | cientos de árboles distintos, y se promedian | un árbol solo es muy inestable |
| **XGBoost** | árboles en fila, cada uno corrige al anterior | exprimir la última gota de precisión |

Y cerramos el **Módulo 1** comparando los siete modelos que vimos en las Clases 2, 3 y 4.

## Qué vas a hacer hoy

| Bloque | Contenido | Min |
|---|---|---|
| 1 | Árboles de regresión: cómo cortan y cómo frenarlos | 15 |
| 2 | Random Forest: promediar árboles distintos | 12 |
| 3 | XGBoost: árboles que corrigen errores | 13 |
| 4 | La comparativa del Módulo 1 | 12 |

Igual que siempre: completa las celdas **`TU CÓDIGO`** y ejecuta la **`VERIFICACIÓN`** de abajo.

---
## Celda 0 · Preparación (idéntica a las Clases 2 y 3)

Misma carga, mismo corte 70/30 estratificado por zona, misma mediana para rellenar `total_bedrooms`
y mismo one-hot de `ocean_proximity`. Al final tienes `Xd_train`, `Xd_test`, `y_train`, `y_test`.

**Novedad:** además separamos un 20 % del entrenamiento como **validación** (`X_ent`, `X_val`).
Ahí vamos a elegir los hiperparámetros, sin tocar la prueba.

XGBoost viene instalado en Colab. Si trabajas en tu computadora y no lo tienes:
`pip install xgboost`.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVR
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings("ignore")

SEED = 42
NAVY, BLUE, MAG, GREEN, GRAY = "#0A2559", "#1A56E8", "#E6115E", "#12B886", "#5B6B87"
plt.rcParams.update({
    "figure.figsize": (7, 4.2), "figure.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
})

def check(nombre, obtenido, esperado, tol=1e-3):
    if obtenido is None:
        print(f"[ ] {nombre}: todavía no calculaste nada")
        return False
    ok = abs(float(obtenido) - float(esperado)) <= tol
    print(f"{'[OK]' if ok else '[X ]'} {nombre}")
    print(f"     tu resultado: {float(obtenido):.4f}   |   esperado: {float(esperado):.4f}")
    return ok

def check_bool(nombre, cond, pista=""):
    print(f"{'[OK]' if cond else '[X ]'} {nombre}")
    if not cond and pista:
        print(f"     pista: {pista}")
    return bool(cond)

def metricas(y_real, y_pred):
    return {"R2": r2_score(y_real, y_pred),
            "MAE": mean_absolute_error(y_real, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_real, y_pred))}

URL = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
try:
    casas = pd.read_csv(URL)
except Exception:
    casas = pd.read_csv("california_housing.csv")

X_todo = casas.drop(columns=["median_house_value"])
y_todo = casas["median_house_value"]
X_train, X_test, y_train, y_test = train_test_split(
    X_todo, y_todo, test_size=0.3, random_state=SEED, stratify=casas["ocean_proximity"])

mediana_dorm = X_train["total_bedrooms"].median()
X_train, X_test = X_train.copy(), X_test.copy()
X_train["total_bedrooms"] = X_train["total_bedrooms"].fillna(mediana_dorm)
X_test["total_bedrooms"] = X_test["total_bedrooms"].fillna(mediana_dorm)

Xd_train = pd.get_dummies(X_train, columns=["ocean_proximity"], drop_first=True, dtype=int)
Xd_test = pd.get_dummies(X_test, columns=["ocean_proximity"], drop_first=True, dtype=int)
COLS = list(Xd_train.columns)

# Validación: 20 % del entrenamiento, para elegir hiperparámetros
X_ent, X_val, y_ent, y_val = train_test_split(Xd_train, y_train, test_size=0.2, random_state=SEED)

# El modelo de la Clase 2, para comparar
ols = LinearRegression().fit(Xd_train, y_train)
m_ols = metricas(y_test, ols.predict(Xd_test))
print(f"entrenamiento {Xd_train.shape} · prueba {Xd_test.shape}")
print(f"  dentro del entrenamiento: {len(X_ent)} para entrenar · {len(X_val)} para validar")
print(f"Regresión lineal de la Clase 2 → R² {m_ols['R2']:.4f} · MAE {m_ols['MAE']:,.0f}")

---
# Bloque 1 · Árboles de regresión  ·  15 min

En la Clase 1 vimos un árbol de **clasificación**: preguntaba «¿ratio de deuda > 0.47?» y en cada hoja
decía «mora» o «no mora». Un árbol de **regresión** hace las mismas preguntas, pero en cada hoja
predice un **número**: el **promedio** de los distritos que cayeron ahí.

¿Cómo elige qué pregunta hacer? Prueba todos los cortes posibles de todas las variables y se queda con
el que deja los dos grupos **más parejos por dentro**, es decir, con la menor suma de errores al cuadrado:

$$\text{SSE}_{\text{corte}} = \sum_{\text{izquierda}} (y_i - \bar{y}_{izq})^2 \; + \sum_{\text{derecha}} (y_i - \bar{y}_{der})^2$$

| Símbolo | Se lee | Qué es |
|---|---|---|
| SSE | «ese ese e» | suma de errores al cuadrado (*sum of squared errors*) |
| $\bar{y}_{izq}$ | «ye barra izquierda» | el promedio del grupo de la izquierda: lo que el árbol predice ahí |

Es la misma suma de cuadrados de la regresión lineal. La diferencia es que aquí cada grupo predice
**una constante** (su promedio), no una recta.

### Ejercicio 1.1 — El primer corte, a mano

Seis distritos, con su ingreso (decenas de miles de USD) y su valor (miles de USD). Los cortes
candidatos son los puntos medios entre ingresos vecinos: 2.5, 3.5, 5, 6.5 y 7.5.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
ingreso = np.array([2, 3, 4, 6, 7, 8])
valor = np.array([100, 120, 110, 250, 270, 290])
cortes = [2.5, 3.5, 5, 6.5, 7.5]

def sse(v):
    return ((v - v.mean()) ** 2).sum()

sse_por_corte = []
for c in cortes:
    izq, der = valor[ingreso <= c], valor[ingreso > c]
    sse_por_corte.append(sse(izq) + sse(der))

mejor_corte = cortes[int(np.argmin(sse_por_corte))]
pred_45 = valor[ingreso <= mejor_corte].mean()

print(f"Sin cortar (predecir siempre el promedio {valor.mean():.0f}): SSE = {sse(valor):,.0f}\n")
print(f"{'corte':>6} | {'prom. izq':>9} | {'prom. der':>9} | {'SSE total':>9}")
for c, s_ in zip(cortes, sse_por_corte):
    marca = "  ← el mejor" if c == mejor_corte else ""
    print(f"{c:>6} | {valor[ingreso <= c].mean():>9.0f} | {valor[ingreso > c].mean():>9.0f} | {s_:>9,.0f}{marca}")
print(f"\nUn distrito con ingreso 4.5 cae a la izquierda: el árbol predice {pred_45:.0f} mil USD.")

In [ ]:
# ── VERIFICACIÓN 1.1 ─────────────────────────────────────────────────────
ok = len(sse_por_corte) == 5 and None not in sse_por_corte
r = [check_bool("calculaste los 5 SSE", ok, "sse_por_corte.append(sse(izq) + sse(der))"),
     check("SSE del corte en 5", sse_por_corte[2] if ok else None, 1000),
     check("mejor corte", mejor_corte, 5, tol=0),
     check("predicción para ingreso 4.5", pred_45, 110)]
print()
print("1.1 OK" if all(r) else "Revisa 1.1")

El corte en 5 baja el SSE de 39 400 a 1 000: separa limpiamente los distritos
baratos de los caros. Un árbol real hace esto mismo con **todas** las variables y **todos** los cortes
posibles, elige el mejor, y luego **repite dentro de cada grupo**. Por eso se dice que es un
algoritmo *voraz* (*greedy*): en cada paso toma el mejor corte del momento, sin mirar hacia adelante.

### Ejercicio 1.2 — Un árbol de dos niveles con los datos reales

Entrena `DecisionTreeRegressor(max_depth=2, random_state=SEED)` con `Xd_train` y mira sus reglas con
`export_text(arbol, feature_names=COLS)`.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
arbol2 = DecisionTreeRegressor(max_depth=2, random_state=SEED).fit(Xd_train, y_train)
m_arbol2 = metricas(y_test, arbol2.predict(Xd_test))

print(export_text(arbol2, feature_names=COLS))
print(f"Cuatro hojas, cuatro precios posibles. R² en prueba {m_arbol2['R2']:.4f} · MAE {m_arbol2['MAE']:,.0f}")

In [ ]:
# ── VERIFICACIÓN 1.2 ─────────────────────────────────────────────────────
r = [check("R² del árbol de 2 niveles en prueba", None if m_arbol2 is None else m_arbol2["R2"], 0.4932),
     check_bool("tiene 4 hojas", arbol2 is not None and arbol2.get_n_leaves() == 4)]
print()
print("1.2 OK" if all(r) else "Revisa 1.2")

**Lee las reglas.** La primera pregunta es el **ingreso** (corte en 5.04, unos 50 mil USD al año): la
misma variable que dominaba la regresión lineal. Pero en la segunda pregunta pasa algo que la recta
no podía hacer: **para los distritos de ingreso bajo pregunta si están tierra adentro** (`INLAND`), y
para los de ingreso alto vuelve a preguntar por el ingreso. Cada rama hace su propia pregunta.

Eso es una **interacción**: el efecto de estar tierra adentro depende del ingreso. El árbol las
encuentra solo, sin que tengas que crear la variable `ingreso × INLAND`.

### Demostración 1.3 — Partir el mapa en regiones

Para ver qué hace un árbol, le damos **solo dos variables**: longitud y latitud. Cada hoja es entonces un
rectángulo del mapa, y todos los distritos que caen en él reciben el mismo valor: el promedio del
rectángulo. Cada punto del gráfico es un distrito de prueba, pintado con lo que predice el árbol.

In [ ]:
# ── DEMOSTRACIÓN 1.3 ─────────────────────────────────────────────────────
geo = ["longitude", "latitude"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.8), sharey=True)
for ax, prof in zip(axes, [2, 4, 8]):
    a = DecisionTreeRegressor(max_depth=prof, random_state=SEED).fit(Xd_train[geo], y_train)
    pred = a.predict(Xd_test[geo])
    sc = ax.scatter(X_test["longitude"], X_test["latitude"], c=pred, cmap="viridis", s=4,
                    vmin=50000, vmax=450000)
    ax.set_title(f"profundidad {prof} · {a.get_n_leaves()} regiones\nR² {r2_score(y_test, pred):.3f}")
    ax.set_xlabel("longitud")
fig.colorbar(sc, ax=axes, label="valor que predice el árbol (USD)", shrink=0.8)
axes[0].set_ylabel("latitud")
plt.show()

Con 2 niveles, 4 rectángulos: casi no dice nada (R² 0.144). Con 8 niveles, 241 rectángulos que
dibujan la costa de Los Ángeles y la bahía de San Francisco: R² 0.619 **usando solo las coordenadas**,
casi lo mismo que la regresión lineal con las 12 variables (0.651).

Es lo que anunciamos al final de la Clase 3: el SVR buscaba la precisión **curvándose**; el árbol la
busca **partiendo el mapa**.

### Ejercicio 1.4 — ¿Cuántos niveles? Elegir con validación

Un árbol sin límite crece hasta dejar **un distrito por hoja**: memoriza. Prueba profundidades de 1 a 20,
entrena con `X_ent`, mide el R² en entrenamiento y en **validación**, y elige la mejor según validación.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
profundidades = list(range(1, 21))
r2_ent, r2_val = [], []
for p in profundidades:
    a = DecisionTreeRegressor(max_depth=p, random_state=SEED).fit(X_ent, y_ent)
    r2_ent.append(r2_score(y_ent, a.predict(X_ent)))
    r2_val.append(r2_score(y_val, a.predict(X_val)))

mejor_prof = profundidades[int(np.argmax(r2_val))]

libre = DecisionTreeRegressor(random_state=SEED).fit(X_ent, y_ent)
print(f"Sin límite: {libre.get_depth()} niveles y {libre.get_n_leaves():,} hojas para {len(X_ent):,} distritos")
print(f"  R² entrenamiento {r2_score(y_ent, libre.predict(X_ent)):.3f} · validación {r2_score(y_val, libre.predict(X_val)):.3f}\n")
print(f"Mejor profundidad por validación: {mejor_prof} → R² validación {max(r2_val):.4f}")

fig, ax = plt.subplots()
ax.plot(profundidades, r2_ent, "o-", color=NAVY, label="entrenamiento")
ax.plot(profundidades, r2_val, "o-", color=MAG, label="validación")
ax.axvline(mejor_prof, color=GRAY, ls="--", lw=1)
ax.set_xlabel("profundidad del árbol (complejidad →)"); ax.set_ylabel("R²")
ax.set_title("La curva de la Clase 1, ahora en regresión")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ── VERIFICACIÓN 1.4 ─────────────────────────────────────────────────────
ok = len(r2_val) == 20 and None not in r2_val
r = [check_bool("guardaste los 20 R² de validación", ok, "r2_val.append(r2_score(y_val, a.predict(X_val)))"),
     check("mejor profundidad", mejor_prof, 9, tol=0),
     check("R² de validación de esa profundidad", max(r2_val) if ok else None, 0.7158)]
print()
print("1.4 OK" if all(r) else "Revisa 1.4")

### Demostración 1.5 — Otro freno: un mínimo de distritos por hoja

`max_depth` corta el árbol parejo, a la misma altura en todas las ramas. Otra opción es dejarlo crecer
lo que quiera, pero **exigir que cada hoja tenga al menos `min_samples_leaf` distritos**. Así las ramas
con muchos datos se dividen más y las de pocos datos paran antes.

In [ ]:
# ── DEMOSTRACIÓN 1.5 ─────────────────────────────────────────────────────
print(f"{'mín. por hoja':>13} | {'hojas':>6} | {'R² entrena':>10} | {'R² valida':>9}")
res_hoja = []
for m in [1, 5, 10, 20, 50, 100]:
    a = DecisionTreeRegressor(min_samples_leaf=m, random_state=SEED).fit(X_ent, y_ent)
    rv = r2_score(y_val, a.predict(X_val)); res_hoja.append((rv, m))
    print(f"{m:>13} | {a.get_n_leaves():>6,} | {r2_score(y_ent, a.predict(X_ent)):>10.3f} | {rv:>9.3f}")
mejor_hoja = max(res_hoja)[1]

# El árbol final: el mejor freno, reentrenado con TODO el entrenamiento, medido una vez en prueba
arbol = DecisionTreeRegressor(min_samples_leaf=mejor_hoja, random_state=SEED).fit(Xd_train, y_train)
m_arbol = metricas(y_test, arbol.predict(Xd_test))
print(f"\nElegido: min_samples_leaf = {mejor_hoja}  ({arbol.get_n_leaves()} hojas)")
print(f"En PRUEBA: R² {m_arbol['R2']:.4f} · MAE {m_arbol['MAE']:,.0f}")

Con `min_samples_leaf = 20` el árbol llega a R² 0.739 en validación, más que con la mejor
profundidad (0.716). En prueba: **R² 0.743 y MAE 38 999**. Un solo árbol ya le gana a la
regresión lineal y queda muy cerca del SVR de la Clase 3.

### Demostración 1.6 — ¿Hay que escalar?

En la Clase 3, escalar era obligatorio para Ridge, Lasso y SVR. Un árbol solo compara cada variable
**consigo misma** («¿ingreso ≤ 5.04?»): cambiar la unidad mueve el umbral, pero no cambia qué distritos
quedan a cada lado.

In [ ]:
# ── DEMOSTRACIÓN 1.6 ─────────────────────────────────────────────────────
sin_escalar = DecisionTreeRegressor(max_depth=9, random_state=SEED).fit(Xd_train, y_train)
con_escalar = make_pipeline(StandardScaler(), DecisionTreeRegressor(max_depth=9, random_state=SEED)).fit(Xd_train, y_train)
pa, pb = sin_escalar.predict(Xd_test), con_escalar.predict(Xd_test)
print(f"R² sin escalar {r2_score(y_test, pa):.4f} · con escalar {r2_score(y_test, pb):.4f}")
print(f"Predicciones idénticas: {100*np.mean(np.isclose(pa, pb)):.1f} %")
print("Las pocas diferencias vienen de empates numéricos al ubicar el umbral, no de la escala.")

---
# Bloque 2 · Random Forest: promediar árboles distintos  ·  12 min

Los árboles tienen un defecto serio: son **inestables**. Cambia un poco los datos y el primer corte
puede ser otro, y con él todo lo que cuelga debajo. Es el problema de **varianza** de la Clase 1.

La solución de Leo Breiman (2001) fue no confiar en un árbol, sino en **cientos**, y promediarlos:

1. **Bootstrap**: cada árbol se entrena con una muestra *con reemplazo* del mismo tamaño que el
   entrenamiento. Algunos distritos salen repetidos y cerca de un tercio no sale.
2. **Variables al azar**: en cada corte, el árbol solo puede elegir entre una parte de las variables
   (`max_features`). Así los árboles no hacen todos la misma primera pregunta.
3. **Promedio**: la predicción del bosque es el promedio de todos los árboles.

Los árboles se dejan crecer **profundos**: cada uno sobreajusta a su manera, y al promediar, sus
errores se cancelan entre sí.

### Demostración 2.1 — Cinco árboles malos, un promedio bueno

In [ ]:
# ── DEMOSTRACIÓN 2.1 ─────────────────────────────────────────────────────
rng = np.random.default_rng(SEED)
predicciones = []
print(f"{'árbol':>5} | {'R² prueba':>9} | {'predice para el distrito 1':>26}")
for k in range(5):
    idx = rng.integers(0, len(Xd_train), len(Xd_train))          # muestra con reemplazo
    a = DecisionTreeRegressor(random_state=SEED).fit(Xd_train.iloc[idx], y_train.iloc[idx])
    p = a.predict(Xd_test); predicciones.append(p)
    print(f"{k+1:>5} | {r2_score(y_test, p):>9.3f} | {p[0]:>26,.0f}")
promedio = np.mean(predicciones, axis=0)
print(f"{'prom.':>5} | {r2_score(y_test, promedio):>9.3f} | {promedio[0]:>26,.0f}")
print(f"\nValor real del distrito 1: {y_test.iloc[0]:,.0f}")

Cada árbol suelto saca un R² de entre 0.60 y 0.65, y para el mismo distrito predicen
valores muy distintos. **El promedio de solo cinco sube a 0.781.** Esa es toda la idea del Random Forest.

### Ejercicio 2.2 — Un bosque de 200 árboles

Entrena `RandomForestRegressor(n_estimators=200, max_features=0.5, oob_score=True, n_jobs=-1,
random_state=SEED)` con `Xd_train`. Tarda unos 10 segundos.

`max_features=0.5` significa que en cada corte el árbol elige entre la mitad de las variables, al azar.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
t0 = time.time()
bosque = RandomForestRegressor(n_estimators=200, max_features=0.5, oob_score=True,
                               n_jobs=-1, random_state=SEED).fit(Xd_train, y_train)
t_bosque = time.time() - t0
r2_oob = bosque.oob_score_
m_rf = metricas(y_test, bosque.predict(Xd_test))

print(f"Entrenado en {t_bosque:.1f} s")
print(f"R² fuera de la bolsa (OOB) = {r2_oob:.4f}")
print(f"R² en prueba               = {m_rf['R2']:.4f} · MAE {m_rf['MAE']:,.0f}")

In [ ]:
# ── VERIFICACIÓN 2.2 ─────────────────────────────────────────────────────
r = [check("R² OOB", r2_oob, 0.8214, tol=3e-3),
     check("R² del bosque en prueba", None if m_rf is None else m_rf["R2"], 0.8280, tol=3e-3),
     check("MAE del bosque en prueba", None if m_rf is None else m_rf["MAE"], 31518.3324, tol=300)]
print()
print("2.2 OK" if all(r) else "Revisa 2.2")

### Validación gratis: el error fuera de la bolsa (OOB)

Cada árbol dejó fuera de su muestra a un tercio de los distritos (la probabilidad de no salir nunca en
$n$ sorteos con reemplazo es $(1 - 1/n)^n \approx 1/e \approx 0.37$). Para cada distrito se promedian
**solo los árboles que no lo vieron**, y eso da una estimación honesta sin apartar validación.

El OOB (0.821) y la prueba (0.828) casi coinciden: el OOB funciona como validación.

### Demostración 2.3 — ¿Cuántos árboles hacen falta?

Agregamos árboles de a poco (`warm_start=True` reutiliza los que ya están) y miramos el OOB.

In [ ]:
# ── DEMOSTRACIÓN 2.3 ─────────────────────────────────────────────────────
creciente = RandomForestRegressor(n_estimators=10, max_features=0.5, oob_score=True, warm_start=True,
                                  n_jobs=-1, random_state=SEED)
cantidades, oobs = [10, 25, 50, 100, 150, 200], []
for n in cantidades:
    creciente.set_params(n_estimators=n)
    creciente.fit(Xd_train, y_train)
    oobs.append(creciente.oob_score_)
    print(f"{n:>4} árboles → R² OOB {creciente.oob_score_:.4f}")

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(cantidades, oobs, "o-", color=GREEN)
ax.set_xlabel("número de árboles"); ax.set_ylabel("R² OOB")
ax.set_title("Más árboles nunca empeora: solo deja de mejorar")
plt.tight_layout(); plt.show()

A diferencia de la profundidad, **el número de árboles no produce overfitting**: más árboles solo
promedian mejor. Lo que cuesta es tiempo. De 100 a 200 árboles el OOB sube apenas
0.004: a partir de ahí, la mejora no paga el tiempo.

¿Y `max_features`? Con todas las variables en cada corte (`max_features=1.0`, que es *bagging* puro)
el bosque saca R² 0.820 en prueba y tarda el doble. Con la mitad, 0.828: árboles más distintos
entre sí se cancelan mejor los errores.

---
# Bloque 3 · XGBoost: árboles que corrigen errores  ·  13 min

Random Forest entrena sus árboles **en paralelo**, cada uno por su lado. El **boosting** los entrena
**en fila**: cada árbol nuevo se entrena para predecir **los errores que dejaron los anteriores**.

1. Empieza prediciendo el promedio para todos.
2. Calcula los residuos: $e_i = y_i - \hat{y}_i$.
3. Entrena un árbol **chico** para predecir esos residuos.
4. Suma a la predicción **una fracción** de lo que dice ese árbol: $\hat{y} \leftarrow \hat{y} + \eta \cdot \text{árbol}$.
5. Vuelve al paso 2.

| Símbolo | Se lee | Qué es |
|---|---|---|
| $\eta$ | «eta» | la **tasa de aprendizaje** (`learning_rate`): qué tan grande es cada paso |

**XGBoost** (*eXtreme Gradient Boosting*) es una implementación muy eficiente de esta idea, con
regularización incluida. En la Clase 15 la veremos a fondo junto a LightGBM y CatBoost.

### Ejercicio 3.1 — Boosting a mano

Los mismos 6 distritos del ejercicio 1.1. Cada «árbol» es un solo corte en ingreso = 5 (un *tocón*,
o *stump*), que predice el **residuo promedio** de cada lado. Usa $\eta = 0.5$ y haz dos rondas.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
eta = 0.5
izq = ingreso <= 5

def un_paso(pred):
    """Un árbol de un corte entrenado sobre los residuos, y el paso de tamaño eta."""
    residuo = valor - pred
    correccion = np.where(izq, residuo[izq].mean(), residuo[~izq].mean())
    return pred + eta * correccion

pred_0 = np.full(6, valor.mean())
pred_1 = un_paso(pred_0)
pred_2 = un_paso(pred_1)
sse_2 = ((valor - pred_2) ** 2).sum()

for k, p in enumerate([pred_0, pred_1, pred_2]):
    print(f"ronda {k}: predice {p[0]:.0f} (izq) y {p[-1]:.0f} (der) · SSE {((valor - p) ** 2).sum():>6,.0f}")
print("\nCada ronda recorre la MITAD de la distancia que falta hasta 110 y 270.")
print("Con eta = 1 llegaría de un salto; con eta chico avanza despacio, pero se equivoca menos al corregir.")

In [ ]:
# ── VERIFICACIÓN 3.1 ─────────────────────────────────────────────────────
r = [check("ronda 1, izquierda", None if pred_1 is None else pred_1[0], 150),
     check("ronda 1, derecha", None if pred_1 is None else pred_1[-1], 230),
     check("SSE después de la ronda 2", sse_2, 3400)]
print()
print("3.1 OK" if all(r) else "Revisa 3.1")

### Ejercicio 3.2 — XGBoost con parada temprana

En boosting **sí** hay overfitting si agregas demasiados árboles: después de corregir la señal, empieza
a corregir el ruido. ¿Cuántos poner? Le damos un máximo alto (3 000) y le pedimos que **pare cuando la
validación lleve 50 árboles sin mejorar** (`early_stopping_rounds=50`).

1. Entrena con `X_ent` mirando `eval_set=[(X_val, y_val)]`, y lee `best_iteration`.
2. Reentrena con **todo** el entrenamiento y ese número de árboles. Mide en prueba una vez.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
buscador = XGBRegressor(n_estimators=3000, learning_rate=0.05, max_depth=6,
                        early_stopping_rounds=50, random_state=SEED)
buscador.fit(X_ent, y_ent, eval_set=[(X_val, y_val)], verbose=False)

n_arboles = buscador.best_iteration + 1
r2_val_xgb = r2_score(y_val, buscador.predict(X_val))

t0 = time.time()
xgb_final = XGBRegressor(n_estimators=n_arboles, learning_rate=0.05, max_depth=6,
                         random_state=SEED).fit(Xd_train, y_train)
t_xgb = time.time() - t0
m_xgb = metricas(y_test, xgb_final.predict(Xd_test))

print(f"La validación dejó de mejorar en el árbol {n_arboles} (R² validación {r2_val_xgb:.4f})")
print(f"Reentrenado con todo en {t_xgb:.1f} s")
print(f"En PRUEBA: R² {m_xgb['R2']:.4f} · MAE {m_xgb['MAE']:,.0f}")

In [ ]:
# ── VERIFICACIÓN 3.2 ─────────────────────────────────────────────────────
r = [check_bool("la parada temprana encontró un número razonable de árboles",
                n_arboles is not None and 300 <= n_arboles <= 2000, "n_arboles = buscador.best_iteration + 1"),
     check("R² de validación", r2_val_xgb, 0.8340, tol=5e-3),
     check("R² de XGBoost en prueba", None if m_xgb is None else m_xgb["R2"], 0.8412, tol=5e-3),
     check("MAE de XGBoost en prueba", None if m_xgb is None else m_xgb["MAE"], 30076.9298, tol=600)]
print()
print("3.2 OK" if all(r) else "Revisa 3.2")

### Demostración 3.3 — Pasos grandes o pasos chicos

La curva de validación del buscador, y lo que pasa si la tasa de aprendizaje es 0.3 en vez de 0.05.

In [ ]:
# ── DEMOSTRACIÓN 3.3 ─────────────────────────────────────────────────────
if n_arboles is not None:
    rapido = XGBRegressor(n_estimators=3000, learning_rate=0.3, max_depth=6,
                          early_stopping_rounds=50, random_state=SEED)
    rapido.fit(X_ent, y_ent, eval_set=[(X_val, y_val)], verbose=False)

    fig, ax = plt.subplots(figsize=(8, 4))
    for m, c, nombre in [(rapido, MAG, "η = 0.3"), (buscador, BLUE, "η = 0.05")]:
        rmse_val = m.evals_result()["validation_0"]["rmse"]
        ax.plot(rmse_val, color=c, lw=1.8, label=f"{nombre} · para en {m.best_iteration + 1} árboles")
    ax.set_ylim(40000, 80000)
    ax.set_xlabel("número de árboles"); ax.set_ylabel("RMSE en validación (USD)")
    ax.set_title("Pasos chicos: más árboles, pero un error más bajo")
    ax.legend(); plt.tight_layout(); plt.show()

    print(f"η = 0.3  → {rapido.best_iteration + 1:>4} árboles · R² validación {r2_score(y_val, rapido.predict(X_val)):.4f}")
    print(f"η = 0.05 → {buscador.best_iteration + 1:>4} árboles · R² validación {r2_val_xgb:.4f}")

Con $\eta = 0.3$ llega rápido y se estanca en R² 0.818. Con $\eta = 0.05$ necesita
981 árboles, pero llega más lejos (0.834). Es el trade-off típico del boosting: **tasa
chica y muchos árboles**, con la parada temprana decidiendo cuántos.

| | Random Forest (bagging) | XGBoost (boosting) |
|---|---|---|
| Los árboles se entrenan | en paralelo, independientes | en fila, cada uno corrige al anterior |
| Cada árbol es | profundo (sobreajusta solo) | chico (subajusta solo) |
| Lo que reduce | la **varianza** | el **sesgo** |
| ¿Más árboles sobreajusta? | no | **sí**: hace falta parada temprana |
| Hiperparámetros delicados | pocos | varios (η, profundidad, n.° de árboles) |

---
# Bloque 4 · La comparativa del Módulo 1  ·  12 min

Siete modelos en tres clases, todos con el mismo corte y la misma prueba. Los hiperparámetros de cada
uno se eligieron con validación (o validación cruzada), nunca con la prueba.

### Demostración 4.1 — Reentrenar los modelos de las Clases 2 y 3

Regresión lineal, Ridge α = 10, Lasso α = 1 000 y el SVR RBF con C = 1000 y ε = 20 (los que eligió la
validación en la Clase 3). Tarda unos segundos.

In [ ]:
# ── DEMOSTRACIÓN 4.1 ─────────────────────────────────────────────────────
m_ridge = metricas(y_test, make_pipeline(StandardScaler(), Ridge(alpha=10)).fit(Xd_train, y_train).predict(Xd_test))
m_lasso = metricas(y_test, make_pipeline(StandardScaler(), Lasso(alpha=1000, max_iter=100000))
                   .fit(Xd_train, y_train).predict(Xd_test))

muestra = np.random.default_rng(SEED).choice(len(Xd_train), 5000, replace=False)
svr = make_pipeline(StandardScaler(), SVR(kernel="rbf", C=1000, epsilon=20))
svr.fit(Xd_train.iloc[muestra], y_train.iloc[muestra] / 1000)
m_svr = metricas(y_test, svr.predict(Xd_test) * 1000)

for nombre, m in [("Ridge", m_ridge), ("Lasso", m_lasso), ("SVR RBF", m_svr)]:
    print(f"{nombre:>8} → R² {m['R2']:.4f} · MAE {m['MAE']:,.0f}")

### Ejercicio 4.2 — La tabla del módulo

Completa el diccionario con los tres modelos de hoy (`m_arbol`, `m_rf`, `m_xgb`), arma la tabla
ordenada por MAE y responde: ¿cuál gana, y cuántos dólares de error por distrito le ahorra a la
regresión lineal?

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
resultados = {
    "Regresión lineal": m_ols,
    "Ridge": m_ridge,
    "Lasso": m_lasso,
    "SVR RBF": m_svr,
    "Árbol": m_arbol,
    "Random Forest": m_rf,
    "XGBoost": m_xgb,
}
tabla = pd.DataFrame(resultados).T.sort_values("MAE")
mejor_modelo = tabla.index[0]
ahorro_mae = m_ols["MAE"] - tabla["MAE"].iloc[0]

print(f"{'modelo':>17} | {'R²':>6} | {'MAE':>7} | {'RMSE':>7}")
for nombre, fila in tabla.iterrows():
    print(f"{nombre:>17} | {fila['R2']:>6.3f} | {fila['MAE']:>7,.0f} | {fila['RMSE']:>7,.0f}")
print(f"\nGana {mejor_modelo}: {ahorro_mae:,.0f} USD menos de error por distrito que la regresión lineal")

fig, ax = plt.subplots(figsize=(8, 4))
colores = [MAG if n == mejor_modelo else (GREEN if n in ("Árbol", "Random Forest") else BLUE) for n in tabla.index]
ax.barh(tabla.index[::-1], tabla["MAE"][::-1], color=colores[::-1])
for i, v in enumerate(tabla["MAE"][::-1]):
    ax.text(v + 500, i, f"{v:,.0f}", va="center", fontsize=10)
ax.set_xlabel("MAE en prueba (USD)  ←  menos es mejor")
ax.set_title("Módulo 1: siete modelos, la misma prueba")
plt.tight_layout(); plt.show()

In [ ]:
# ── VERIFICACIÓN 4.2 ─────────────────────────────────────────────────────
ok = None not in resultados.values()
r = [check_bool("completaste los 7 modelos", ok, "Árbol: m_arbol · Random Forest: m_rf · XGBoost: m_xgb"),
     check_bool("el mejor modelo es XGBoost", mejor_modelo == "XGBoost"),
     check("ahorro de MAE frente a la regresión lineal", ahorro_mae, 19587.1533, tol=800)]
print()
print("Bloque 4 · 4.2 OK" if all(r) else "Revisa 4.2")

XGBoost baja el error típico de **49 664 a 30 077 USD por distrito**, un 39 % menos que la
recta. Random Forest queda muy cerca (31 518) con un hiperparámetro menos de qué preocuparse.

> **Una advertencia honesta.** Elegir al ganador mirando la prueba es, en sí mismo, usar la prueba para
> decidir. Con siete modelos y diferencias grandes, el riesgo es bajo. Si las diferencias fueran de
> décimas, el ganador habría que elegirlo con validación cruzada (Clase 13) y dejar la prueba solo
> para confirmar.

### Demostración 4.3 — ¿Qué variables usa XGBoost? Depende de a quién le preguntes

Dos formas de medir importancia:

- **Por ganancia** (`feature_importances_`): cuánto bajó el error, sumando todos los cortes donde se usó
  la variable. Sale gratis del entrenamiento.
- **Por permutación**: desordena una columna de la prueba al azar y mide **cuánto sube el MAE**. Si el
  modelo no la necesitaba, casi no sube.

In [ ]:
# ── DEMOSTRACIÓN 4.3 ─────────────────────────────────────────────────────
if m_xgb is not None:
    perm = permutation_importance(xgb_final, Xd_test, y_test, n_repeats=5, random_state=SEED,
                                  scoring="neg_mean_absolute_error")
    imp = pd.DataFrame({"ganancia (%)": 100 * xgb_final.feature_importances_,
                        "permutación (+USD de MAE)": perm.importances_mean}, index=COLS)
    imp = imp.sort_values("permutación (+USD de MAE)", ascending=False)
    print(imp.round({"ganancia (%)": 1, "permutación (+USD de MAE)": 0}).to_string())

Las dos medidas cuentan historias muy distintas:

- **Por ganancia**, `INLAND` es lo más importante (57 %). Es el primer gran corte del
  árbol y se lleva el crédito de toda esa reducción de error.
- **Por permutación**, lo más importante es **la ubicación**: desordenar la longitud o la latitud sube
  el MAE en unos 67 967 USD. Desordenar `INLAND` cuesta poco (9 910), porque las coordenadas ya
  dicen si un distrito está tierra adentro.
- `ISLAND` tiene un 6 % de la ganancia y desordenarla no cuesta nada: con 5 distritos, la ganancia la
  sobrevalora.

> La importancia por ganancia dice **qué usó** el modelo para entrenar; la de permutación, **qué
> necesita** para predecir. Para explicar un modelo a un comité, la segunda es más honesta.

### Demostración 4.4 — Dónde se equivoca cada uno

Los residuos de la regresión lineal y de XGBoost sobre el mapa, con la misma escala que en la Clase 3.

In [ ]:
# ── DEMOSTRACIÓN 4.4 ─────────────────────────────────────────────────────
if m_xgb is not None:
    res_ols = y_test.values - ols.predict(Xd_test)
    res_xgb = y_test.values - xgb_final.predict(Xd_test)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
    for ax, res, titulo in [(axes[0], res_ols, "Regresión lineal"), (axes[1], res_xgb, "XGBoost")]:
        sc = ax.scatter(X_test["longitude"], X_test["latitude"], c=np.clip(res, -150000, 150000),
                        cmap="coolwarm", s=4, vmin=-150000, vmax=150000)
        ax.set_title(f"{titulo} · MAE {np.mean(np.abs(res)):,.0f}")
        ax.set_xlabel("longitud")
    axes[0].set_ylabel("latitud")
    fig.colorbar(sc, ax=axes, label="residuo (USD)", shrink=0.8)
    plt.show()

Las manchas de la regresión lineal casi desaparecen: XGBoost aprendió la geografía partiendo el mapa
en miles de rectángulos. Quedan 296 distritos con errores de más de 100 mil USD, y casi la mitad
(47 %) valen más de 350 mil, aunque esos distritos son solo el 12 % de la prueba.
Los caros son los más difíciles, en parte porque el valor real está topado en 500 001.

### Precisión, interpretabilidad y costo

| Modelo | R² | ¿Escalar? | ¿Se puede leer? | Ojo con |
|---|---|---|---|---|
| Regresión lineal | 0.651 | no hace falta | **sí**: un coeficiente por variable | supuestos, multicolinealidad |
| Ridge / Lasso | 0.651 / 0.647 | **sí** | sí | elegir α |
| SVR RBF | 0.759 | **sí** | no | lento con muchas filas |
| Árbol | 0.743 | no | **sí**, si es chico | inestable; sobreajusta sin freno |
| Random Forest | 0.828 | no | solo con importancias | pesado en memoria |
| XGBoost | **0.841** | no | solo con importancias | más hiperparámetros; sobreajusta sin parada temprana |

Los modelos de árboles **no extrapolan**: nunca predicen fuera del rango que vieron en entrenamiento.
Aquí no importa (el valor está topado), pero si los precios suben por encima de todo lo visto, un árbol
seguirá prediciendo el techo antiguo.

> En riesgos, el patrón habitual es usar el modelo interpretable como **modelo oficial** y el de
> árboles como **modelo retador** (*challenger*): si el retador gana por mucho, hay una señal que el
> oficial no está capturando y vale la pena buscarla.

---
# Cierre

### Los números del Módulo 1 (en prueba)

| Clase | Modelo | R² | MAE |
|---|---|---|---|
| 2 | Regresión lineal | 0.651 | 49 664 |
| 3 | Ridge α = 10 | 0.651 | 49 653 |
| 3 | Lasso α = 1 000 | 0.647 | 49 973 |
| 3 | SVR RBF (C = 1000, ε = 20) | 0.759 | 37 926 |
| 4 | Árbol (mín. 20 por hoja) | 0.743 | 38 999 |
| 4 | Random Forest (200 árboles) | 0.828 | 31 518 |
| 4 | **XGBoost (parada temprana)** | **0.841** | **30 077** |

### Checklist de salida

- [ ] Sé cómo elige un árbol sus cortes y qué predice en cada hoja.
- [ ] Freno un árbol con `max_depth` o `min_samples_leaf`, elegidos con validación.
- [ ] Sé por qué un árbol no necesita escalar las variables.
- [ ] Sé por qué promediar árboles distintos reduce la varianza, y qué es el OOB.
- [ ] Sé que el boosting entrena árboles en fila sobre los residuos, y por qué necesita parada temprana.
- [ ] Distingo importancia por ganancia de importancia por permutación.
- [ ] Puedo defender qué modelo del módulo usaría y por qué, más allá del R².

### Reto para quien termine antes

Cambia en XGBoost `max_depth=6` por 3 y por 10 (con la misma parada temprana). ¿Cuál gana en
validación? ¿Y cuántos árboles necesita cada uno?

### Próxima clase · Miércoles 7/10

**Clase 5 · Métricas de clasificación y regresión logística.** Empieza el Módulo 2: dejamos de predecir
dólares y volvemos a predecir sí o no. La exactitud de la Clase 1 se va a quedar corta muy rápido.